<a href="https://colab.research.google.com/github/shayanR10/FIV1/blob/FIV1.2/FIV1(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#prerequisites, run this cell first; make sure to also create a Google Kaggle Account along with Colab secrets named "KAGGLE_USERNAME" , "KAGGLE_SLUG" , and "KAGGLE_KEY" for your Google Kaggle
# username, dataset slug, and custom API key, respectively.

!pip install ftfy regex tqdm -q
!pip install kagglehub -q
!pip install numpy -q
!pip install safetensors -q
!pip install cowsay -q
!pip install pillow -q
!pip install open3d -q
!pip install ultralytics -q
!pip install open_clip_torch -q
!pip install transformers -q
!pip install morphosource -q
!pip install trimesh -q

In [ ]:
#Kaggle connection setup; data pipeline

import os
from google.colab import userdata
import kagglehub

username = userdata.get('KAGGLE_USERNAME')
slug = userdata.get('KAGGLE_SLUG')
key = userdata.get('KAGGLE_KEY')
os.environ["KAGGLE_USERNAME"] = username
os.environ["KAGGLE_KEY"] = key

print("Authenticating")

try:
    path = kagglehub.dataset_download(f"{username}/{slug}")
    print("Success")
    print(path)
except Exception as e:
    print("Failed.")
    kagglehub.login()

In [ ]:
# sam2 (segment anything model)

import os
from PIL import Image, ImageOps
import numpy as np
from ultralytics import SAM
from google.colab import userdata
import kagglehub

print("Waking up SAM 2...")

kUser = userdata.get('KAGGLE_USERNAME')
kSlug = userdata.get('KAGGLE_SLUG')
datapath = kagglehub.dataset_download(f"{kUser}/{kSlug}")

sam = SAM("sam2.1_t.pt")

maskedoutputdir = os.path.join(os.path.dirname(datapath), "fossil_masked_dataset")
os.makedirs(maskedoutputdir, exist_ok=True)

print(f"Processing images from {datapath}...")

processed_count = 0
skipped_count = 0

for root, dirs, files in os.walk(datapath):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(root, file)
            print(f"-> Processing: {file}")

            try:
                results = sam(img_path, verbose=False)

                if results and results[0].masks is not None:
                    mask = results[0].masks.data[0].cpu().numpy()

                    img = Image.open(img_path).convert("RGBA")
                    img_np = np.array(img)

                    mask_resized = Image.fromarray((mask * 255).astype(np.uint8)).resize(img.size, Image.Resampling.NEAREST)
                    mask_np = np.array(mask_resized)

                    img_np[:, :, 3] = mask_np
                    masked_img = Image.fromarray(img_np, "RGBA")

                    rel_path = os.path.relpath(img_path, datapath)
                    base_name = os.path.splitext(rel_path)[0]
                    out_path = os.path.join(maskedoutputdir, base_name + ".png")

                    os.makedirs(os.path.dirname(out_path), exist_ok=True)
                    masked_img.save(out_path)
                    processed_count += 1
                else:

                    skipped_count += 1
            except Exception as e:
                print(f"Error processing {file}: {e}. Skipping...")
                skipped_count += 1
                continue

print(f"\nSAM 2 background stripping complete! Successfully processed {processed_count} images (Skipped/No mask: {skipped_count}).")

In [ ]:
# automated multi view rendering for 3d scans

import open3d as o3d
import numpy as np
import math
import os

def classify_fossil_morphology(mesh_path):
    mesh = o3d.io.read_triangle_mesh(mesh_path)

    if mesh.is_empty() or len(mesh.vertices) == 0:
        print(f"Warning: mesh is empty or unreadable: {mesh_path}")
        return "skull_or_vertebrae"

    bbox = mesh.get_axis_aligned_bounding_box()
    extents = np.asarray(bbox.get_extents(), dtype=np.float64)

    if extents.shape != (3,) or not np.isfinite(extents).all():
        print(f"Warning: invalid bounding box: {mesh_path}")
        return "skull_or_vertebrae"

    positive_extents = extents[extents > 1e-9]
    if len(positive_extents) == 0:
        print(f"Warning: degenerate bounding box: {mesh_path}")
        return "skull_or_vertebrae"

    max_extent = float(np.max(positive_extents))
    normalized_extents = np.zeros(3, dtype=np.float64)

    if len(positive_extents) == 3:
        normalized_extents = positive_extents / max_extent
    else:
        normalized_extents[:len(positive_extents)] = positive_extents / max_extent
        normalized_extents = np.sort(normalized_extents)

    normalized_extents = np.sort(normalized_extents)
    min_axis = float(normalized_extents[0])
    mid_axis = float(normalized_extents[1])
    max_axis = float(normalized_extents[2])

    if min_axis > 1e-9:
        aspect_ratio = max_axis / min_axis
    else:
        aspect_ratio = float("inf")

    vertex_count = len(mesh.vertices)
    triangle_count = len(mesh.triangles)
    topology_size = vertex_count + triangle_count

    # Very sparse, small-complexity, or strongly elongated assets are isolated elements.
    if vertex_count < 5000:
        return "isolated_element"

    if aspect_ratio >= 3.2 and vertex_count < 12000:
        return "isolated_element"

    if aspect_ratio >= 2.8 and min_axis < 0.35:
        return "isolated_element"

    # High-complexity, expansive assets are whole skeletons.
    if vertex_count >= 50000 or topology_size >= 75000:
        return "whole_skeleton"

    if vertex_count >= 25000 and mid_axis >= 0.45 and aspect_ratio < 4.0:
        return "whole_skeleton"

    # Remaining intermediate-resolution assets are treated as skulls or vertebrae.
    return "skull_or_vertebrae"

In [ ]:
"""
dinosaur specimen scraper for the triple-stream classifier.
Sources:
  - Paleobiology Database (PBDB) for taxonomic alignment
  - GBIF + iDigBio Network for licensed specimen occurrence images
  - MorphoSource for licensed 3D fossil and scan meshes

Outputs:
  /root/raw_dino_staging/
    images/<accepted_taxon>/
    meshes/isolated_element/
    meshes/skull_or_vertebrae/
    meshes/whole_skeleton/
    manifests/<source>_*.jsonl
    manifests/staging_manifest.json
    dataset-metadata.json
"""
import argparse
import gc
import hashlib
import json
import logging
import os
import re
import shutil
from collections import defaultdict
from pathlib import Path
from urllib.parse import quote, urlparse, urlunparse
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

try:
    from PIL import Image, UnidentifiedImageError
except ImportError:
    Image = None
    UnidentifiedImageError = IOError

try:
    import open3d as o3d
except ImportError:
    o3d = None

try:
    import numpy as np
except ImportError:
    np = None

STAGING_ROOT = Path("/root/raw_dino_staging")
PBDB_URL = "https://paleobiodb.org/data1.2/taxa/list.json"
GBIF_BASE = "https://api.gbif.org/v1"
MORPHOSOURCE_BASE = "https://www.morphosource.org"

DEFAULT_CORE_DINOS = [
    "Stegosaurus", "Kentrosaurus", "Hesperosaurus", "Tuojiangosaurus",
    "Miragaia", "Dacentrurus", "Edmontosaurus", "Saurolophus",
    "Maiasaura", "Brachylophosaurus", "Gryposaurus", "Prosaurolophus",
    "Shantungosaurus", "Parasaurolophus", "Corythosaurus",
    "Lambeosaurus", "Hypacrosaurus", "Olorotitan", "Tsintaosaurus",
    "Dryosaurus", "Ouranosaurus", "Iguanodon", "Mantellisaurus",
    "Protohadros", "Probactrosaurus", "Bactrosaurus", "Eolambia",
    "Tethyshadros", "Telmatosaurus", "Diplodocus", "Apatosaurus",
    "Brontosaurus", "Dicraeosaurus", "Amargasaurus", "Camarasaurus",
    "Giraffatitan", "Brachiosaurus", "Patagotitan", "Dreadnoughtus",
    "Rapetosaurus", "Saltasaurus", "Futalognkosaurus", "Shunosaurus",
    "Mamenchisaurus", "Ankylosaurus", "Zuul", "Euoplocephalus",
    "Pinacosaurus", "Saichania", "Tarchia", "Anodontosaurus",
    "Scolosaurus", "Akainacephalus", "Gobisaurus", "Shamosaurus",
    "Jinyunpelta", "Baryonyx", "Suchomimus", "Spinosaurus",
    "Ceratosuchops", "Riparovenator", "Irritator", "Spinomimus",
    "Dromaeosaurus", "Utahraptor", "Deinonychus", "Achillobator",
    "Velociraptor", "Tsaagan", "Linheraptor", "Microraptor",
    "Graciliraptor", "Sinornithosaurus", "Changyuraptor",
    "Buitreraptor", "Halszkaraptor", "Sinornithoides", "Gobivenator",
    "Mei", "Coelophysis", "Procompsognathus", "Dilophosaurus",
    "Ceratosaurus", "Carnotaurus", "Majungasaurus", "Megaraptor",
    "Allosaurus", "Sinraptor", "Monolophosaurus", "Acrocanthosaurus",
    "Giganotosaurus", "Carcharodontosaurus", "Tyrannosaurus",
    "Tarbosaurus", "Albertosaurus", "Gorgosaurus", "Daspletosaurus",
    "Yutyrannus", "Guanlong", "Compsognathus", "Sinosauropteryx",
    "Sinocalliopteryx", "Huaxiagnathus", "Gallimimus",
    "Struthiomimus", "Ornithomimus", "Therizinosaurus", "Alxasaurus",
    "Erlikosaurus", "Anzu", "Citipati", "Khaan", "Conchoraptor",
    "Caudipteryx", "Triceratops", "Torosaurus", "Chasmosaurus",
    "Anchiceratops", "Arrhinoceratops", "Kosmoceratops",
    "Utahceratops", "Pentaceratops", "Centrosaurus",
    "Styracosaurus", "Pachyrhinosaurus", "Nasutoceratops",
    "Einiosaurus", "Diabloceratops", "Wendiceratops",
    "Medusaceratops", "Sinoceratops", "Protoceratops", "Psittacosaurus"
]

OPEN_LICENSE_ALLOWLIST = (
    "cc0",
    "cc-by",
    "creativecommons.org/publicdomain/zero",
    "creativecommons.org/licenses/by",
)

MESH_EXTENSIONS = (".ply", ".obj", ".stl", ".glb")
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".webp", ".tif", ".tiff")

SIGLIP_MODEL_NAME = "google/siglip2-base-patch16-224"
SIGLIP_FOSSIL_CONFIDENCE = 0.15

DEFAULT_SIGLIP_POSITIVE_PROMPTS = [
    "a photo of a fossil dinosaur bone, tooth, or skull of {taxon}",
    "a photo of a museum dinosaur skeleton exhibit of {taxon}",
    "a close-up scientific photograph of a paleontological fossil specimen of {taxon}",
    "a complete fossil skeleton of {taxon}",
]

DEFAULT_SIGLIP_NEGATIVE_PROMPTS = [
    "a photo of a map, chart, graph, or museum label card",
    "a photo of a landscape, rock outcrop without bones, or modern animal",
    "a drawing, illustration, or CGI render",
]

siglip_processor = None
siglip_model = None
siglip_device = None


def setup_logging():
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        force=True,
    )


def get_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return None


def build_session(user_agent):
    retries = Retry(
        total=2,
        connect=1,
        read=2,
        status=2,
        backoff_factor=0.5,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["GET", "POST", "HEAD"]),
        raise_on_status=False,
    )
    session = requests.Session()
    session.headers.update({
        "User-Agent": user_agent,
        "Accept": "application/json,text/plain,*/*",
    })
    session.mount("https://", HTTPAdapter(max_retries=retries))
    session.mount("http://", HTTPAdapter(max_retries=retries))
    return session


def normalize_taxon_name(name):
    if not name:
        return ""
    cleaned = re.sub(r"\s+", " ", str(name).strip())
    cleaned = re.sub(r"\b(?:sp\.?|indet\.?|cf\.?)\b", "", cleaned, flags=re.I)
    cleaned = re.sub(r"[^A-Za-z0-9 _-]", "", cleaned).strip()
    parts = cleaned.split()
    if not parts:
        return ""
    if len(parts) == 1:
        return parts[0].capitalize()
    return f"{parts[0].capitalize()} {parts[1].lower()}"


def safe_folder_name(name):
    normalized = normalize_taxon_name(name)
    normalized = re.sub(r"[^A-Za-z0-9_-]+", "_", normalized)
    return normalized.strip("_") or "Unidentified_Dinosauria"


def license_text(value):
    if isinstance(value, dict):
        return " ".join(str(v) for v in value.values())
    if isinstance(value, (list, tuple, set)):
        return " ".join(str(v) for v in value)
    return str(value or "")


def is_open_license(value, allow_nc=False):
    text = license_text(value).lower()
    if not allow_nc and ("-nc" in text or "/by-nc" in text or "noncommercial" in text):
        return False
    if "-nd" in text or "/by-nd" in text:
        return False
    return any(token in text for token in OPEN_LICENSE_ALLOWLIST)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def clean_url(url):
    parts = urlparse(url.strip())
    safe_path = quote(parts.path, safe="/:@!$&'()*+,;=-._~%")
    return urlunparse((parts.scheme, parts.netloc, safe_path, parts.params, parts.query, parts.fragment))


def file_extension(url, content_type="", default=".bin"):
    path_ext = Path(urlparse(url).path).suffix.lower()
    if path_ext:
        return path_ext
    content_type = content_type.lower()
    if "jpeg" in content_type:
        return ".jpg"
    if "png" in content_type:
        return ".png"
    if "webp" in content_type:
        return ".webp"
    if "polygon" in content_type:
        return ".ply"
    if "stl" in content_type:
        return ".stl"
    if "obj" in content_type:
        return ".obj"
    if "glb" in content_type:
        return ".glb"
    return default


def stream_download(session, url, destination, max_bytes, extra_headers=None):
    destination = Path(destination)
    if destination.exists() and destination.stat().st_size > 0:
        return True
    tmp_path = destination.with_suffix(destination.suffix + ".part")
    tmp_path.unlink(missing_ok=True)
    headers = dict(extra_headers or {})
    url = clean_url(url)
    try:
        with session.get(url, headers=headers, timeout=(10, 45), stream=True) as response:
            response.raise_for_status()
            content_length = int(response.headers.get("Content-Length", "0") or "0")
            if content_length > max_bytes:
                logging.warning("Skipping oversized download %s (%d bytes)", url, content_length)
                return False
            destination.parent.mkdir(parents=True, exist_ok=True)
            downloaded = 0
            with open(tmp_path, "wb") as output:
                for chunk in response.iter_content(chunk_size=1024 * 128):
                    if not chunk:
                        continue
                    downloaded += len(chunk)
                    if downloaded > max_bytes:
                        logging.warning("Aborting oversized download %s", url)
                        tmp_path.unlink(missing_ok=True)
                        return False
                    output.write(chunk)
        if tmp_path.stat().st_size == 0:
            tmp_path.unlink(missing_ok=True)
            return False
        os.replace(tmp_path, destination)
        return True
    except requests.RequestException as exc:
        logging.warning("Download failed %s: %s", url, exc)
        tmp_path.unlink(missing_ok=True)
        return False
    except Exception as exc:
        logging.warning("Unexpected download error %s: %s", url, exc)
        tmp_path.unlink(missing_ok=True)
        return False


def append_jsonl(path, record):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False, sort_keys=True) + "\n")


def fetch_pbdb(session, timeout=30):
    params = {
        "base_name": "Dinosauria",
        "status": "accepted",
        "limit": "all",
        "show": "class",
    }
    response = session.get(PBDB_URL, params=params, timeout=timeout)
    response.raise_for_status()
    payload = response.json()
    records = payload.get("records", [])
    dinotax2 = {}
    for record in records:
        txnID = record.get("oid")
        if not txnID:
            continue
        dinotax2[txnID] = {
            "name": record.get("nam"),
            "rank": record.get("rnk"),
            "parent": record.get("par"),
        }
    logging.info("PBDB returned %d accepted Dinosauria taxa", len(dinotax2))
    return dinotax2


def build_accepted_taxa(dinotax2, core_dinos, limit):
    by_genus = set()
    for record in dinotax2.values():
        name = normalize_taxon_name(record.get("name"))
        rank = (record.get("rank") or "").lower()
        if name and rank in ("genus", "species"):
            by_genus.add(name.split()[0].lower())
    accepted = []
    for candidate in core_dinos:
        normalized = normalize_taxon_name(candidate)
        genus = normalized.split()[0].lower()
        if genus in by_genus and normalized not in accepted:
            accepted.append(normalized)
            if len(accepted) >= limit:
                break
    if len(accepted) < limit:
        for record in dinotax2.values():
            rank = (record.get("rank") or "").lower()
            name = normalize_taxon_name(record.get("name"))
            if rank != "genus" or not name:
                continue
            genus = name.split()[0]
            if genus not in accepted:
                accepted.append(genus)
                if len(accepted) >= limit:
                    break
    logging.info("PBDB-aligned species: %d", len(accepted))
    return accepted


def verify_image(path):
    if Image is None:
        return False
    try:
        with Image.open(path) as image:
            image.verify()
        with Image.open(path) as image:
            image.convert("RGB")
        return True
    except (UnidentifiedImageError, IOError, SyntaxError, ValueError, RuntimeError) as exc:
        logging.warning("Removing invalid image %s: %s", path, exc)
        try:
            path.unlink(missing_ok=True)
        except Exception:
            pass
        return False


def load_siglip2():
    global siglip_processor, siglip_model, siglip_device
    if siglip_model is not None and siglip_processor is not None:
        return True
    try:
        import torch
        from transformers import AutoModel, AutoProcessor
        siglip_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        siglip_processor = AutoProcessor.from_pretrained(
            SIGLIP_MODEL_NAME,
            trust_remote_code=True,
        )
        siglip_model = AutoModel.from_pretrained(
            SIGLIP_MODEL_NAME,
            trust_remote_code=True,
        ).to(siglip_device)
        siglip_model.eval()
        logging.info("SigLIP 2 loaded on %s: %s", siglip_device, SIGLIP_MODEL_NAME)
        return True
    except Exception as exc:
        logging.error("SigLIP 2 initialization failed; downloaded images will be rejected: %s", exc)
        return False


def is_relevant_fossil_image(
    image_path,
    taxon_name,
    confidence_threshold=SIGLIP_FOSSIL_CONFIDENCE,
    positive_prompts=None,
    negative_prompts=None,
):
    if not load_siglip2():
        return False, 0.0
    try:
        import torch
        cleaned_taxon = normalize_taxon_name(taxon_name)
        pos_templates = positive_prompts or DEFAULT_SIGLIP_POSITIVE_PROMPTS
        neg_templates = negative_prompts or DEFAULT_SIGLIP_NEGATIVE_PROMPTS
        pos_labels = [p.format(taxon=cleaned_taxon) for p in pos_templates]
        neg_labels = [n.format(taxon=cleaned_taxon) for n in neg_templates]
        labels = pos_labels + neg_labels
        with Image.open(image_path) as opened_image:
            image = opened_image.convert("RGB")
            inputs = siglip_processor(
                text=labels,
                images=[image],
                return_tensors="pt",
                padding="max_length",
                max_length=64,
            )
            inputs = {name: tensor.to(siglip_device) for name, tensor in inputs.items()}
            with torch.no_grad():
                outputs = siglip_model(**inputs)
                logits = outputs.logits_per_image[0].detach().float().cpu()
                # SigLIP 2 uses independent sigmoid probabilities per prompt
                probs = torch.sigmoid(logits)
        num_pos = len(pos_labels)
        best_pos = float(probs[:num_pos].max().item())
        best_neg = float(probs[num_pos:].max().item())
        is_relevant = best_pos >= confidence_threshold and best_pos > best_neg
        return is_relevant, best_pos
    except Exception as exc:
        logging.warning("SigLIP 2 relevance check failed for %s; rejecting image: %s", image_path, exc)
        return False, 0.0


def save_verified_image(
    session,
    url,
    destination,
    max_bytes,
    taxon_name,
    confidence_threshold=SIGLIP_FOSSIL_CONFIDENCE,
    positive_prompts=None,
):
    temporary = destination.with_suffix(destination.suffix + ".download")
    temporary.unlink(missing_ok=True)
    if not stream_download(session, url, temporary, max_bytes):
        return False, 0.0
    if not verify_image(temporary):
        return False, 0.0
    relevant, score = is_relevant_fossil_image(
        temporary,
        taxon_name,
        confidence_threshold=confidence_threshold,
        positive_prompts=positive_prompts,
    )
    destination.parent.mkdir(parents=True, exist_ok=True)
    if relevant and Image is not None:
        try:
            with Image.open(temporary) as image:
                rgb = image.convert("RGB")
                rgb.save(destination, format="JPEG", quality=92, optimize=True)
            temporary.unlink(missing_ok=True)
            return True, score
        except Exception as exc:
            logging.warning("Image conversion failed for %s: %s", url, exc)
    else:
        logging.info(
            "  [SigLIP Rejected] %s | score=%.3f (threshold=%.2f) | %s",
            taxon_name,
            score,
            confidence_threshold,
            url,
        )
    temporary.unlink(missing_ok=True)
    return False, score


def flatten_media(media):
    if not isinstance(media, list):
        media = [media]
    return [item for item in media if isinstance(item, dict)]


def scrape_gbif(
    session,
    taxon,
    dest_dir,
    max_items,
    max_bytes,
    manifest_path,
    confidence_threshold=SIGLIP_FOSSIL_CONFIDENCE,
    positive_prompts=None,
):
    search_url = f"{GBIF_BASE}/occurrence/search"
    params = {
        "scientificName": taxon,
        "mediaType": "StillImage",
        "basisOfRecord": "FOSSIL_SPECIMEN",
        "limit": 300,
    }
    saved = 0
    offset = 0
    seen_urls = set()
    while saved < max_items:
        params["offset"] = offset
        try:
            response = session.get(search_url, params=params, timeout=(10, 30))
            response.raise_for_status()
            payload = response.json()
        except requests.RequestException as exc:
            logging.warning("GBIF/iDigBio search failed for %s: %s", taxon, exc)
            break
        results = payload.get("results", [])
        if not results:
            break
        for occurrence in results:
            latitude = occurrence.get("decimalLatitude")
            longitude = occurrence.get("decimalLongitude")
            for media in flatten_media(occurrence.get("media")):
                if saved >= max_items:
                    break
                url = media.get("identifier") or media.get("references")
                if not url or url in seen_urls:
                    continue
                seen_urls.add(url)
                license_value = media.get("license") or occurrence.get("license")
                if not is_open_license(license_value):
                    continue
                ext = file_extension(url, default=".jpg")
                if ext not in IMAGE_EXTENSIONS:
                    continue
                record_hash = hashlib.sha256(
                    f"gbif_merged:{occurrence.get('key')}:{url}".encode()
                ).hexdigest()[:16]
                filename = f"{record_hash}.jpg"
                destination = dest_dir / filename
                if destination.exists() and verify_image(destination):
                    relevant, score = is_relevant_fossil_image(
                        destination,
                        taxon,
                        confidence_threshold=confidence_threshold,
                        positive_prompts=positive_prompts,
                    )
                    if relevant:
                        saved += 1
                        logging.info(
                            "  [Image #%d Verified Existing] %s -> %s (SigLIP=%.3f)",
                            saved,
                            taxon,
                            filename,
                            score,
                        )
                    else:
                        destination.unlink(missing_ok=True)
                    continue
                ok, score = save_verified_image(
                    session,
                    url,
                    destination,
                    max_bytes,
                    taxon_name=taxon,
                    confidence_threshold=confidence_threshold,
                    positive_prompts=positive_prompts,
                )
                if not ok:
                    continue
                record = {
                    "source": "GBIF+iDigBio",
                    "taxon": taxon,
                    "license": license_value,
                    "source_url": url,
                    "occurrence_url": f"https://www.gbif.org/occurrence/{occurrence.get('key')}",
                    "record_id": str(occurrence.get("key") or ""),
                    "dataset_key": occurrence.get("datasetKey"),
                    "institution_code": occurrence.get("institutionCode"),
                    "catalog_number": occurrence.get("catalogNumber"),
                    "scientific_name": occurrence.get("scientificName") or taxon,
                    "decimal_latitude": latitude,
                    "decimal_longitude": longitude,
                    "siglip_score": round(score, 4),
                    "local_path": str(destination),
                    "sha256": sha256_file(destination),
                }
                append_jsonl(manifest_path, record)
                saved += 1
                logging.info(
                    "  [Image #%d Saved] %s -> %s (SigLIP=%.3f | %s)",
                    saved,
                    taxon,
                    filename,
                    score,
                    occurrence.get("institutionCode") or "GBIF",
                )
        if len(results) < params["limit"]:
            break
        offset += params["limit"]
    return saved


def morphosource_item_license(item):
    if isinstance(item, dict):
        candidates = [
            item.get("license"),
            item.get("licenses"),
            item.get("rights"),
            item.get("rights_statement"),
            item.get("dcterms:license"),
        ]
    else:
        candidates = [
            getattr(item, "license", None),
            getattr(item, "licenses", None),
            getattr(item, "rights", None),
            getattr(item, "rights_statement", None),
        ]
    return " ".join(license_text(c) for c in candidates if c)


def morphosource_item_extension(item):
    if isinstance(item, dict):
        possible = [
            item.get("original_filename"),
            item.get("file_name"),
            item.get("title"),
            item.get("format"),
            item.get("media_type"),
        ]
    else:
        possible = [
            getattr(item, "original_filename", None),
            getattr(item, "file_name", None),
            getattr(item, "title", None),
            getattr(item, "format", None),
            getattr(item, "media_type", None),
        ]
    for value in possible:
        if not value:
            continue
        lower = str(value).lower()
        for ext in MESH_EXTENSIONS:
            if ext in lower:
                return ext
    return None


def detect_mesh_extension(file_path, fallback_ext=None):
    try:
        with open(file_path, "rb") as f:
            head = f.read(512).lstrip()
        if not head or head.startswith(b"<"):
            return None
        if head.startswith(b"ply"):
            return ".ply"
        if head.startswith(b"glTF"):
            return ".glb"
        if head.startswith(b"solid "):
            return ".stl"
        text_head = head.decode("utf-8", errors="ignore").lstrip()
        if any(line.startswith(("v ", "vn ", "f ", "o ", "g ", "# ")) for line in text_head.splitlines()[:10]):
            return ".obj"
        if len(head) >= 84:
            return ".stl"
    except Exception:
        pass
    return fallback_ext


def scrape_morphosource(
    session,
    morpho_api_key,
    taxon,
    mesh_root,
    max_items,
    max_bytes,
    manifest_path,
):
    if not morpho_api_key:
        logging.warning("MorphoSource API key unavailable; skipping MorphoSource")
        return 0
    import zipfile
    try:
        from morphosource import DownloadConfig, DownloadVisibility, search_media as ms_search_media
    except Exception as exc:
        logging.warning("MorphoSource import failed (%s); installing now...", exc)
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "morphosource"])
        from morphosource import DownloadConfig, DownloadVisibility, search_media as ms_search_media
    download_config = DownloadConfig(
        api_key=morpho_api_key,
        use_statement=(
            "Scientific dinosaur fossil 3D morphology research, "
            "classification dataset preparation, and non-commercial model evaluation."
        ),
        use_categories=["Research"],
    )
    saved = 0
    page = 1
    seen_ids = set()
    while saved < max_items and page <= 10:
        try:
            results = ms_search_media(
                taxon,
                media_type="Mesh",
                visibility=DownloadVisibility.OPEN,
                per_page=50,
                page=page,
            )
        except Exception as exc:
            logging.warning("MorphoSource package search failed for %s: %s", taxon, exc)
            break
        items = getattr(results, "items", None) or []
        if not items:
            break
        for item in items:
            if saved >= max_items:
                break
            media_id = str(
                item.get("id") if isinstance(item, dict) else getattr(item, "id", "")
            ).strip()
            if not media_id or media_id in seen_ids:
                continue
            seen_ids.add(media_id)
            license_value = morphosource_item_license(item)
            if license_value and not is_open_license(license_value, allow_nc=True):
                continue
            ext = morphosource_item_extension(item)
            if ext and ext not in MESH_EXTENSIONS:
                continue
            identifier_hash = hashlib.sha256(
                f"morphosource:{media_id}".encode()
            ).hexdigest()[:16]
            incoming_dir = mesh_root / "_incoming"
            incoming_dir.mkdir(parents=True, exist_ok=True)
            zip_path = incoming_dir / f"{identifier_hash}.zip"
            try:
                item.download_bundle(str(zip_path), download_config)
            except Exception as exc:
                logging.warning("MorphoSource download failed for %s: %s", media_id, exc)
                zip_path.unlink(missing_ok=True)
                continue
            if not zip_path.exists() or zip_path.stat().st_size == 0:
                zip_path.unlink(missing_ok=True)
                continue
            extracted_mesh = None
            if zipfile.is_zipfile(zip_path):
                try:
                    with zipfile.ZipFile(zip_path, "r") as zf:
                        for member in zf.namelist():
                            member_ext = Path(member).suffix.lower()
                            if member_ext in MESH_EXTENSIONS and not Path(member).name.startswith("."):
                                extracted_mesh = incoming_dir / f"{identifier_hash}_raw{member_ext}"
                                with zf.open(member) as src, open(extracted_mesh, "wb") as dst:
                                    shutil.copyfileobj(src, dst)
                                break
                finally:
                    zip_path.unlink(missing_ok=True)
            else:
                extracted_mesh = incoming_dir / f"{identifier_hash}_raw{ext or '.bin'}"
                os.replace(zip_path, extracted_mesh)
            if not extracted_mesh or not extracted_mesh.exists():
                continue
            if extracted_mesh.stat().st_size == 0 or extracted_mesh.stat().st_size > max_bytes:
                extracted_mesh.unlink(missing_ok=True)
                continue
            zip_member_ext = extracted_mesh.suffix.lower() if extracted_mesh.suffix.lower() in MESH_EXTENSIONS else ext
            actual_ext = detect_mesh_extension(extracted_mesh, fallback_ext=zip_member_ext)
            if not actual_ext or actual_ext not in MESH_EXTENSIONS:
                extracted_mesh.unlink(missing_ok=True)
                continue
            temporary = incoming_dir / f"{identifier_hash}{actual_ext}"
            os.replace(extracted_mesh, temporary)
            category = classify_fossil_mesh(temporary)
            category_dir = mesh_root / category
            category_dir.mkdir(parents=True, exist_ok=True)
            destination = category_dir / f"{identifier_hash}{actual_ext}"
            os.replace(temporary, destination)
            title = item.get("title") if isinstance(item, dict) else getattr(item, "title", "")
            record = {
                "source": "MorphoSource",
                "taxon": taxon,
                "license": license_value or "MorphoSource-Research",
                "source_url": f"{MORPHOSOURCE_BASE}/concern/media/{media_id}",
                "record_id": media_id,
                "title": title,
                "morphology_category": category,
                "local_path": str(destination),
                "sha256": sha256_file(destination),
            }
            append_jsonl(manifest_path, record)
            saved += 1
            logging.info(
                "  [Mesh #%d Saved] %s -> %s/%s%s",
                saved,
                taxon,
                category,
                identifier_hash,
                actual_ext,
            )
        if len(items) < 50:
            break
        page += 1
    return saved


def _stats_from_counts_and_extents(vertex_count, triangle_count, extents):
    if vertex_count == 0 or extents.size != 3 or not np.isfinite(extents).all():
        return None
    positive = extents[extents > 1e-9]
    if positive.size == 0:
        return None
    max_extent = float(np.max(positive))
    normalized = np.clip(positive / max_extent, 0.0, 1.0)
    if normalized.size == 3:
        sorted_extents = np.sort(normalized)
    else:
        sorted_extents = np.pad(normalized, (0, 3 - normalized.size), mode="constant")
    aspect_ratio = 1.0
    if sorted_extents[0] > 1e-9:
        aspect_ratio = float(sorted_extents[-1] / sorted_extents[0])
    return {
        "vertex_count": int(vertex_count),
        "triangle_count": int(triangle_count),
        "normalized_extents": sorted_extents.tolist(),
        "aspect_ratio": aspect_ratio,
    }


def mesh_topology_stats(mesh_path):
    if o3d is not None and np is not None:
        try:
            mesh = o3d.io.read_triangle_mesh(str(mesh_path), enable_post_processing=True)
            if mesh.has_vertices() and len(mesh.vertices) > 0:
                bbox = mesh.get_axis_aligned_bounding_box()
                extents = np.asarray(bbox.get_extent(), dtype=np.float64)
                stats = _stats_from_counts_and_extents(len(mesh.vertices), len(mesh.triangles), extents)
                if stats is not None:
                    return stats
        except Exception:
            pass
    try:
        import trimesh
        tm = trimesh.load(str(mesh_path), force="mesh", process=False)
        if hasattr(tm, "vertices") and len(tm.vertices) > 0:
            extents = np.asarray(tm.extents, dtype=np.float64)
            faces = len(tm.faces) if hasattr(tm, "faces") else 0
            return _stats_from_counts_and_extents(len(tm.vertices), faces, extents)
    except Exception as exc:
        logging.warning("trimesh fallback failed for %s: %s", mesh_path, exc)
    return None


def classify_fossil_mesh(mesh_path):
    if np is None:
        logging.warning("numpy unavailable; defaulting mesh to skull_or_vertebrae")
        return "skull_or_vertebrae"
    try:
        stats = mesh_topology_stats(mesh_path)
        if stats is None:
            return "skull_or_vertebrae"
        vertices = stats["vertex_count"]
        triangles = stats["triangle_count"]
        aspect = stats["aspect_ratio"]
        normalized = stats["normalized_extents"]
        mid_axis = float(normalized[1] if len(normalized) > 1 else 1.0)
        min_axis = float(normalized[0] if normalized else 1.0)
        complexity = vertices + triangles
        if vertices < 5000 and (aspect > 2.8 or min_axis < 0.12):
            return "isolated_element"
        if vertices > 25000 or complexity > 60000:
            return "whole_skeleton"
        if 0.30 <= mid_axis <= 0.95 and 5000 <= vertices <= 30000:
            return "skull_or_vertebrae"
        if aspect > 2.2 and vertices < 12000:
            return "isolated_element"
        if aspect < 1.8 and vertices >= 12000:
            return "whole_skeleton"
        return "skull_or_vertebrae"
    except Exception as exc:
        logging.warning("Mesh classification failed for %s: %s", mesh_path, exc)
        return "skull_or_vertebrae"


def create_dataset_metadata(staging_root, dataset_title):
    metadata = {
        "title": dataset_title,
        "id": "",
        "licenses": [{"name": "CC0-1.0"}, {"name": "CC-BY-4.0"}],
        "description": (
            "Dinosaur fossil specimen images and 3D meshes collected from "
            "GBIF, iDigBio, and MorphoSource under open academic terms."
        ),
        "subtitle": "Taxonomically PBDB-aligned scientific dinosaur dataset",
        "isPrivate": True,
        "resources": [
            {"path": "images", "description": "Licensed specimen images"},
            {"path": "meshes", "description": "Morphology-classified 3D fossil meshes"},
            {"path": "manifests", "description": "Source, license, and provenance manifests"},
        ],
    }
    metadata_path = staging_root / "dataset-metadata.json"
    with open(metadata_path, "w", encoding="utf-8") as handle:
        json.dump(metadata, handle, indent=2, sort_keys=True)
    return metadata_path


def kaggle_sync(staging_root, username, slug, title, sync_mode):
    if not username or not slug:
        logging.warning("Kaggle username/slug unavailable; skipping Kaggle sync")
        return False
    if sync_mode == "none":
        logging.info("Kaggle sync disabled")
        return False
    handle = f"{username}/{slug}"
    metadata_path = create_dataset_metadata(staging_root, title)
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    metadata["id"] = handle
    metadata_path.write_text(json.dumps(metadata, indent=2, sort_keys=True), encoding="utf-8")
    logging.info("Preparing Kaggle dataset staging for %s", handle)
    try:
        import kagglehub
        kagglehub.dataset_upload(
            handle=handle,
            local_dataset_dir=str(staging_root),
            version_notes="fiv2.0 scientific scraper refresh",
        )
        logging.info("Kaggle upload complete: %s", handle)
        return True
    except Exception as exc:
        logging.warning("kagglehub.dataset_upload failed: %s", exc)
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
        if sync_mode == "create":
            api.dataset_create_new(
                folder=str(staging_root),
                public=False,
                quiet=False,
                convert_to_csv=False,
                dir_mode="zip",
            )
        else:
            api.dataset_create_version(
                folder=str(staging_root),
                version_notes="fiv2.0 scientific scraper refresh",
                quiet=False,
                convert_to_csv=False,
                delete_old_versions=False,
                dir_mode="zip",
            )
        logging.info("Kaggle API sync complete: %s", handle)
        return True
    except Exception as exc:
        logging.error("Kaggle API fallback sync failed: %s", exc)
        return False


def write_staging_manifest(staging_root, stats):
    manifest_path = staging_root / "manifests" / "staging_manifest.json"
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(stats, indent=2, sort_keys=True), encoding="utf-8")
    return manifest_path


def main():
    setup_logging()
    parser = argparse.ArgumentParser(description="FIV1 scientific dinosaur scraper")
    parser.add_argument("--max-taxa", type=int, default=200)
    parser.add_argument("--images-per-taxon", type=int, default=200)
    parser.add_argument("--meshes-per-taxon", type=int, default=20)
    parser.add_argument("--max-image-mb", type=int, default=15)
    parser.add_argument("--max-mesh-mb", type=int, default=150)
    parser.add_argument("--staging-root", default=str(STAGING_ROOT))
    parser.add_argument("--dry-run", action="store_true")
    parser.add_argument("--kaggle-sync", choices=("none", "create", "update"), default="none")
    parser.add_argument("--kaggle-title", default="Fossil DINO Dataset fiv2.0")
    parser.add_argument(
        "--siglip-confidence",
        type=float,
        default=SIGLIP_FOSSIL_CONFIDENCE,
        help="Minimum SigLIP 2 fossil probability required for image acceptance",
    )
    parser.add_argument(
        "--siglip-prompts",
        nargs="+",
        default=None,
        help="Custom positive text prompts for SigLIP 2 (supports {taxon} placeholder)",
    )
    args, _ = parser.parse_known_args()
    if not 0.0 < args.siglip_confidence < 1.0:
        parser.error("--siglip-confidence must be between 0.0 and 1.0")
    staging_root = Path(args.staging_root)
    (staging_root / "images").mkdir(parents=True, exist_ok=True)
    (staging_root / "meshes" / "isolated_element").mkdir(parents=True, exist_ok=True)
    (staging_root / "meshes" / "skull_or_vertebrae").mkdir(parents=True, exist_ok=True)
    (staging_root / "meshes" / "whole_skeleton").mkdir(parents=True, exist_ok=True)
    (staging_root / "manifests").mkdir(parents=True, exist_ok=True)
    kUser = get_secret("KAGGLE_USERNAME")
    kSlug = get_secret("KAGGLE_SLUG")
    morpho_api_key = get_secret("MORPHOSOURCE_API_KEY")
    kKey = get_secret("KAGGLE_KEY")
    if kUser:
        os.environ["KAGGLE_USERNAME"] = kUser
    if kKey:
        os.environ["KAGGLE_KEY"] = kKey
    user_agent = f"fiv2.0/1.0 (https://kaggle.com/{kUser or 'unknown'}; scientific fossil dataset)"
    session = build_session(user_agent)
    try:
        dinotax2 = fetch_pbdb(session)
    except Exception as exc:
        logging.critical("PBDB alignment unavailable: %s", exc)
        return 1
    accepted_taxa = build_accepted_taxa(dinotax2, DEFAULT_CORE_DINOS, limit=args.max_taxa)
    stats = {
        "staging_root": str(staging_root),
        "pbdb_taxon_count": len(dinotax2),
        "accepted_taxon_count": len(accepted_taxa),
        "accepted_taxa": accepted_taxa,
        "dry_run": args.dry_run,
        "sources": defaultdict(lambda: {"images": 0, "meshes": 0, "taxa_processed": 0}),
        "taxonomy": {},
    }
    if args.dry_run:
        write_staging_manifest(staging_root, stats)
        logging.info("Dry run complete. Accepted taxa: %d", len(accepted_taxa))
        create_dataset_metadata(staging_root, args.kaggle_title)
        return 0
    if o3d is None:
        logging.warning("Open3D unavailable; mesh classification will use fallback placement")
    max_image_bytes = args.max_image_mb * 1024 * 1024
    max_mesh_bytes = args.max_mesh_mb * 1024 * 1024
    for taxon in accepted_taxa:
        folder_name = safe_folder_name(taxon)
        image_dir = staging_root / "images" / folder_name
        image_dir.mkdir(parents=True, exist_ok=True)
        logging.info("Processing [%s]", taxon)
        gbif_manifest = staging_root / "manifests" / "gbif_records.jsonl"
        morpho_manifest = staging_root / "manifests" / "morphosource_records.jsonl"
        try:
            gbif_count = scrape_gbif(
                session,
                taxon,
                image_dir,
                args.images_per_taxon,
                max_image_bytes,
                gbif_manifest,
                confidence_threshold=args.siglip_confidence,
                positive_prompts=args.siglip_prompts,
            )
            stats["sources"]["GBIF+iDigBio"]["images"] += gbif_count
            stats["sources"]["GBIF+iDigBio"]["taxa_processed"] += 1
        except Exception as exc:
            logging.exception("GBIF/iDigBio processing failed for %s: %s", taxon, exc)
        try:
            morpho_count = scrape_morphosource(
                session,
                morpho_api_key,
                taxon,
                staging_root / "meshes",
                args.meshes_per_taxon,
                max_mesh_bytes,
                morpho_manifest,
            )
            stats["sources"]["MorphoSource"]["meshes"] += morpho_count
            stats["sources"]["MorphoSource"]["taxa_processed"] += 1
        except Exception as exc:
            logging.exception("MorphoSource processing failed for %s: %s", taxon, exc)

        # Periodic garbage collection between taxa to prevent RAM OOM crashes
        gc.collect()
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass

    stats["sources"] = dict(stats["sources"])
    write_staging_manifest(staging_root, stats)
    create_dataset_metadata(staging_root, args.kaggle_title)
    sync_ok = False
    if args.kaggle_sync != "none":
        sync_ok = kaggle_sync(
            staging_root,
            kUser,
            kSlug,
            args.kaggle_title,
            args.kaggle_sync,
        )
    stats["kaggle_sync"] = sync_ok
    write_staging_manifest(staging_root, stats)
    logging.info("scraping complete")
    logging.info("Images and meshes staged at %s", staging_root)
    return 0 if sync_ok or args.kaggle_sync == "none" else 2


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
import json
from pathlib import Path

staging_manifest = Path("/root/raw_dino_staging/manifests/staging_manifest.json")
if staging_manifest.exists():
    stats = json.loads(staging_manifest.read_text(encoding="utf-8"))
    print(json.dumps(stats, indent=2))
else:
    print("Staging manifest not found yet.")

In [ ]:
import kagglehub
from google.colab import userdata

username = userdata.get("KAGGLE_USERNAME")
slug = userdata.get("KAGGLE_SLUG")

kagglehub.dataset_upload(
    handle=f"{username}/{slug}",
    local_dataset_dir="/root/raw_dino_staging",
    version_notes="FIV1 production scientific dataset refresh & SigLIP 2",
)

In [ ]:
# CONFIGURATIONS
imagesizing = (224, 224, 3)    # Image sizing requirement
confidencelevel = (0.85, 0.50) # Confidence thresholds: if confidencelevel, the level of confidence the model has -
                               # in its prediction is 0.85/85% or more, it confirms that exact species. -
                               # Otherwise, graceful fallback is triggered (e.g. Ouranosaurus would simply be -
                               # classified as "unidentified basal hadrosauriform" [or simpler if needbe].)

# safeguard-1: confirms tuples containing image sizing and confidence intervals.
def safeguard1():
  print(imagesizing, confidencelevel)

safeguard1()

# requesting data from the Paleobiology Database's (PBDB) live API
import requests
DINO_INFO_url = "https://paleobiodb.org/data1.2/taxa/list.json?base_name=Dinosauria&status=accepted"
def getdinosauria(DINO_INFO_url):
  response = requests.get(DINO_INFO_url, timeout=30)
  response.raise_for_status()
  dinodata = response.json()
  return dinodata

dinodata = getdinosauria(DINO_INFO_url)

# parses through data
dinotax2 = {}
for records in dinodata["records"]:
  txnID = records["oid"]
  dinotax2[txnID] = {
      "name": records.get("nam"),
      "rank": records.get("rnk"),
      "parent": records.get("par")
  }

def fallingback(txnID, allowed=["genus", "subfamily", "family", "superfamily", "infraorder"]):
  records2 = dinotax2.get(txnID)
  if not records2:
    return "Unidentified Dinosauria", "clade"

  parentID = records2.get("parent")

  bestmatchN = "Unidentified Dinosauria"
  bestmatchR = "clade"

  while parentID:
    parentrnk = dinotax2.get(parentID)
    if not parentrnk:
      break

    currentrank = parentrnk.get("rank")
    currentname = parentrnk.get("name")

    if currentrank in allowed:
      bestmatchN = currentname
      bestmatchR = currentrank
      if currentrank == allowed[-1]:
        break

    parentID = parentrnk.get("parent")

  return bestmatchN, bestmatchR


# safeguard
print(f"{len(dinotax2)} taxons identified")

# conditionals concerning previously defined confidence thresholds
highconf = confidencelevel[0]
lowconf = confidencelevel[1]

def guessing(species, confscore, txnID):
  if confscore >= highconf:
    print(f"{confscore * 100:.1f}% confident of {species}")
    return species, confscore, None
  elif confscore >= lowconf:
    fallbackname, fallbackrank = fallingback(txnID)
    print(f"{confscore * 100:.1f}% confident of {fallbackname} in {fallbackrank}")
    return fallbackname, confscore, fallbackrank
  else:
    fallbackname, fallbackrank = fallingback(txnID)
    print(f"{confscore * 100:.1f}% confidence, unable to identify. Defaulting to broad clade: {fallbackname} ({fallbackrank})")
    return None, confscore, fallbackrank


# neural network, image processing
import torch
import torch.nn as neuralnetwork
import torchvision.models as TVmodels
import torchvision.datasets as dataset
from torchvision import transforms
from torch.utils.data import DataLoader, Subset
import kagglehub
import os
import numpy as nump
import random
from collections import defaultdict
import cowsay
from safetensors.torch import save_file
import json
import open_clip

try:
  import cv2
  from PIL import Image, ImageFilter
except ImportError:
  cv2 = None
  from PIL import Image, ImageFilter

try:
  from google.colab import userdata
except ImportError:
  userdata = None


# pulling data from kaggle dataset for the model to process
kUser = None
kSlug = None
if userdata is not None:
  kUser = userdata.get("KAGGLE_USERNAME")
  kSlug = userdata.get("KAGGLE_SLUG")

if kUser and kSlug:
  print("Fetching dataset from Kaggle")
  datapath = kagglehub.dataset_download(f"{kUser}/{kSlug}")
else:
  datapath = os.environ.get("DINO_DATASET_PATH")

if not datapath:
  raise RuntimeError(
      "Set DINO_DATASET_PATH or Kaggle Colab secrets "
      "KAGGLE_USERNAME/KAGGLE_SLUG."
  )

print(f"Data ready at: {datapath}")


# training optimizations to avoid model from memorizing training images
class Stream2NoiseGuard:
  def __init__(self, kernel_size=3, sigma=0.8, enhance=False):
    self.kernel_size = kernel_size
    self.sigma = sigma
    self.enhance = enhance

  def __call__(self, pil_image):
    image = pil_image.convert("RGB")
    image = image.filter(
        ImageFilter.GaussianBlur(
            radius=(self.sigma * (self.kernel_size - 1) / 2)
        )
    )
    if not self.enhance:
      return image
    if cv2 is None:
      print("[WARNING] cv2 unavailable; Stream 2 enhancement is skipped.")
      return image

    array = nump.asarray(image)
    lab = cv2.cvtColor(array, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8))
    lab = cv2.merge((clahe.apply(l), a, b))
    clahe_rgb = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    laplacian = cv2.Laplacian(
        cv2.cvtColor(clahe_rgb, cv2.COLOR_RGB2GRAY),
        cv2.CV_32F,
        ksize=3
    )
    laplacian = cv2.normalize(
        laplacian,
        None,
        alpha=0.0,
        beta=1.0,
        norm_type=cv2.NORM_MINMAX
    )
    laplacian_rgb = cv2.cvtColor(
        (laplacian * 255).astype(nump.uint8),
        cv2.COLOR_GRAY2RGB
    )
    blended = nump.clip(0.90 * clahe_rgb + 0.10 * laplacian_rgb, 0, 255)
    return Image.fromarray(blended.astype(nump.uint8))


# ViT stream 1 transforms
macrotransforms = transforms.Compose([
  transforms.Resize((224, 224)),
  transforms.RandomHorizontalFlip(p=0.5),
  transforms.ColorJitter(brightness=0.1, contrast=0.1),
  transforms.ToTensor(),
  transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# micro-texture stream 2
microtransforms = transforms.Compose([
  Stream2NoiseGuard(kernel_size=3, sigma=0.8, enhance=True),
  transforms.Resize((256, 256)),
  transforms.RandomResizedCrop(224, scale=(0.3, 0.6)),
  transforms.RandomHorizontalFlip(p=0.5),
  transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
  transforms.ToTensor(),
  transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
  transforms.RandomErasing(p=0.1, scale=(0.02, 0.08), value="random")
])

# 3d scan transforms
spatialtransforms = transforms.Compose([
  transforms.Resize((224, 224)),
  transforms.ToTensor(),
  transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# val transforms
valmacro = transforms.Compose([
  transforms.Resize((224, 224)),
  transforms.ToTensor(),
  transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
valmicro = transforms.Compose([
  Stream2NoiseGuard(kernel_size=3, sigma=0.8, enhance=False),
  transforms.Resize((256, 256)),
  transforms.CenterCrop(224),
  transforms.ToTensor(),
  transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
valspatial = transforms.Compose([
  transforms.Resize((224, 224)),
  transforms.ToTensor(),
  transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class tripledata(torch.utils.data.Dataset):
  def __init__(self, root_dir, macroTf, microTf, spatialTf):
    self.base_dataset = dataset.ImageFolder(root=root_dir)
    self.classes = self.base_dataset.classes
    self.targets = self.base_dataset.targets
    self.macroTf = macroTf
    self.microTf = microTf
    self.spatialTf = spatialTf

  def __len__(self):
    return len(self.base_dataset)

  def __getitem__(self, idx):
    img, label = self.base_dataset[idx]
    return {
        "macro": self.macroTf(img),
        "micro": self.microTf(img),
        "spatial": self.spatialTf(img),
        "label": label
    }

fulltrainset = tripledata(
    root_dir=datapath,
    macroTf=macrotransforms,
    microTf=microtransforms,
    spatialTf=spatialtransforms
)
valset = tripledata(
    root_dir=datapath,
    macroTf=valmacro,
    microTf=valmicro,
    spatialTf=valspatial
)
speciesamt = len(fulltrainset.classes)

with open("class_names.json", "w") as file1:
  json.dump(fulltrainset.classes, file1)

targetranks = ["family", "genus", "superfamily", "subfamily", "infraorder"]

RECORDname = {}
for taxonID2, record in dinotax2.items():
  if record["name"]:
    RECORDname[record["name"].strip().lower()] = record

HIERARCHY = {}
rankvocabularies = {r: {} for r in targetranks}

for cINDx, cNAME in enumerate(fulltrainset.classes):
  cleaned = cNAME.replace("_", " ").strip().lower()
  record = RECORDname.get(cleaned)

  mapped = {}
  currentR = record
  while currentR:
    rank = currentR.get("rank")
    name = currentR.get("name")
    if rank in targetranks and name:
      if name not in rankvocabularies[rank]:
        rankvocabularies[rank][name] = len(rankvocabularies[rank])
      mapped[rank] = rankvocabularies[rank][name]

    parentID2 = currentR.get("parent")
    currentR = dinotax2.get(parentID2) if parentID2 else None

  HIERARCHY[cINDx] = mapped

# fallback sizes if vocabularies are empty
rank_classes = {r: max(1, len(rankvocabularies[r])) for r in targetranks}
print("Hierarchical Head Classes ->", rank_classes)

# splitting: 80% of images go to training while the rest go to validation testing
datasetSIZE = len(fulltrainset)
targets = fulltrainset.targets

classindices = defaultdict(list)
for index, target in enumerate(targets):
  classindices[target].append(index)

trainindices = []
valindices = []

random.seed(42)
for target, indices in classindices.items():
  random.shuffle(indices)

  if len(indices) == 1:
    trainindices.extend(indices)
  else:
    split = max(1, int(len(indices) * 0.8))
    trainindices.extend(indices[:split])
    valindices.extend(indices[split:])

random.shuffle(trainindices)
random.shuffle(valindices)

training = Subset(fulltrainset, trainindices)
validation = Subset(valset, valindices)

# separate validation model selection from post-hoc temperature calibration
generator = torch.Generator().manual_seed(42)
validation_selection, validation_calibration = torch.utils.data.random_split(
    validation,
    [len(validation) // 2, len(validation) - (len(validation) // 2)],
    generator=generator
)

# another safeguard
print(f"{speciesamt} species found for training")
print(f"Training on {len(training)} images, Validating on {len(validation)} images.")

# data loaders
loader = DataLoader(training, batch_size=16, shuffle=True)
validationL = DataLoader(validation_selection, batch_size=16, shuffle=False)
calibrationL = DataLoader(validation_calibration, batch_size=16, shuffle=False)


# triple stream data loaders
class CrossAttentiveFusion(neuralnetwork.Module):
  def __init__(self, embed_dim=256, num_heads=4):
    super().__init__()
    self.MULTI = neuralnetwork.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
    self.lNORM = neuralnetwork.LayerNorm(embed_dim)
    self.gating = neuralnetwork.Sequential(
        neuralnetwork.Linear(embed_dim * 3, 3),
        neuralnetwork.Sigmoid()
    )

  def forward(self, z1, z2, z3):
    sequenceTENSOR = torch.stack([z1, z2, z3], dim=1)
    concatflat = torch.cat([z1, z2, z3], dim=1)
    gate_vals = self.gating(concatflat).unsqueeze(-1)
    attenout, _ = self.MULTI(sequenceTENSOR, sequenceTENSOR, sequenceTENSOR)
    attenout = self.lNORM(attenout + sequenceTENSOR)
    gated = attenout * gate_vals
    fused = gated.reshape(gated.shape[0], -1)
    return fused

class MSDhead(neuralnetwork.Module):
  def __init__(self, in_features, numclasses1, numsamples1=5, p=0.2):
    super().__init__()
    self.numsamples1 = numsamples1
    self.layernorm = neuralnetwork.LayerNorm(in_features)
    self.act = neuralnetwork.GELU()
    self.dropouts = neuralnetwork.ModuleList(
        [neuralnetwork.Dropout(p) for _ in range(numsamples1)]
    )
    self.linear = neuralnetwork.Linear(in_features, numclasses1)

  def forward(self, AT):
    AT = self.layernorm(AT)
    AT = self.act(AT)
    if not self.training:
      return self.linear(AT)
    logits = sum(self.linear(drop(AT)) for drop in self.dropouts) / self.numsamples1
    return logits

class OrthogonalStreamContrastiveLoss(neuralnetwork.Module):
  def __init__(self, alpha=0.1):
    super().__init__()
    self.alpha = alpha

  def forward(self, z1, z2, z3):
    eps = 1e-8

    z1norm = z1 / (torch.norm(z1, p=2, dim=1, keepdim=True) + eps)
    z2norm = z2 / (torch.norm(z2, p=2, dim=1, keepdim=True) + eps)
    z3norm = z3 / (torch.norm(z3, p=2, dim=1, keepdim=True) + eps)

    sim12 = torch.sum(z1norm * z2norm, dim=1).abs().mean()
    sim13 = torch.sum(z1norm * z3norm, dim=1).abs().mean()
    sim23 = torch.sum(z2norm * z3norm, dim=1).abs().mean()

    ortholoss = (sim12 + sim13 + sim23) / 3.0
    return self.alpha * ortholoss


class SAM(torch.optim.Optimizer):
  def __init__(self, params, base_optimizer, rho=0.05, **kwargs):
    assert rho >= 0.0, f"Invalid rho, should be non-negative: {rho}"
    param_groups = params if params and isinstance(params[0], dict) else list(params)
    defaults = dict(rho=rho, **kwargs)
    super(SAM, self).__init__(param_groups, defaults)

    self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
    self.param_groups = self.base_optimizer.param_groups

  @torch.no_grad()
  def first_step(self, zero_grad=False):
    grad_norm = self._grad_norm()

    if not torch.isfinite(grad_norm) or grad_norm <= 1e-12:
      if zero_grad:
        self.zero_grad(set_to_none=True)
      return

    grad_norm = torch.clamp(grad_norm, min=1e-12)
    for group in self.param_groups:
      scale = group["rho"] / grad_norm
      for p in group["params"]:
        if p.grad is None:
          continue
        if not torch.isfinite(p.grad).all():
          continue
        self.state[p]["old_p"] = p.detach().clone()
        epsil = p.grad * scale
        p.add_(epsil)

    if zero_grad:
      self.zero_grad(set_to_none=True)

  @torch.no_grad()
  def second_step(self, zero_grad=False):
    for group in self.param_groups:
      for p in group["params"]:
        old_p = self.state[p].pop("old_p", None)
        if old_p is not None:
          p.copy_(old_p)

    self.base_optimizer.step()
    if zero_grad:
      self.zero_grad(set_to_none=True)

  @torch.no_grad()
  def _grad_norm(self):
    shared_device = self.param_groups[0]["params"][0].device
    finite_grads = []

    for group in self.param_groups:
      for p in group["params"]:
        if p.grad is not None and torch.isfinite(p.grad).all():
          finite_grads.append(p.grad.detach().norm(p=2).to(shared_device))

    if not finite_grads:
      return torch.tensor(0.0, device=shared_device)

    grad_norm = torch.norm(torch.stack(finite_grads), p=2)
    return grad_norm if torch.isfinite(grad_norm) else torch.tensor(
        0.0, device=shared_device
    )

  def zero_grad(self, set_to_none=False):
    self.base_optimizer.zero_grad(set_to_none=set_to_none)


def find_bioclip_blocks(visual):
  if hasattr(visual, "transformer") and hasattr(visual.transformer, "resblocks"):
    return list(visual.transformer.resblocks)
  if hasattr(visual, "blocks"):
    return list(visual.blocks)
  if hasattr(visual, "trunk") and hasattr(visual.trunk, "blocks"):
    return list(visual.trunk.blocks)
  return []

def freeze_bioclip_visual_blocks(visual, freeze_fraction=0.75, top_trainable=3):
  blocks = find_bioclip_blocks(visual)
  if not blocks:
    print("[WARNING] BioCLIP transformer blocks not found; no selective freeze applied.")
    for p in visual.parameters():
      p.requires_grad = False
    return 0

  frozen_count = int(len(blocks) * freeze_fraction)
  frozen_count = max(0, min(frozen_count, len(blocks) - top_trainable))
  for p in visual.parameters():
    p.requires_grad = True

  for block in blocks[:frozen_count]:
    for p in block.parameters():
      p.requires_grad = False
  for block in blocks[frozen_count:]:
    for p in block.parameters():
      p.requires_grad = True

  for name in ("ln_post", "proj", "head"):
    module = getattr(visual, name, None)
    if module is not None:
      for p in module.parameters():
        p.requires_grad = True

  frozen = sum(not p.requires_grad for p in visual.parameters())
  trainable = sum(p.requires_grad for p in visual.parameters())
  print(f"BioCLIP visual: froze {frozen_count}/{len(blocks)} bottom blocks "
        f"({trainable} tensors trainable, {frozen} frozen)")
  return len(blocks) - frozen_count

def build_optimizer_param_groups(model, visual, base_lr=1e-5, decay=0.75, weight_decay=0.01):
  blocks = find_bioclip_blocks(visual)
  block_ids = {id(block): index for index, block in enumerate(blocks)}
  decay_groups = defaultdict(list)

  for name, p in model.named_parameters():
    if not p.requires_grad:
      continue

    lr = base_lr
    parent_block = None
    if name.startswith("path1ViT."):
      for block in blocks:
        if any(p is block_p for block_p in block.parameters()):
          parent_block = block
          break

    if parent_block is not None:
      block_index = block_ids[id(parent_block)]
      depth_from_top = len(blocks) - 1 - block_index
      lr = base_lr * (decay ** depth_from_top)

    is_normalized = name.endswith("bias") or "norm" in name.lower() or "ln" in name.lower()
    decay_groups[(lr, is_normalized)].append(p)

  groups = []
  seen_parameters = set()
  for (lr, is_normalized), parameters in sorted(
      decay_groups.items(), key=lambda item: item[0][0], reverse=True
  ):
    unique = []
    for p in parameters:
      if id(p) not in seen_parameters:
        unique.append(p)
        seen_parameters.add(id(p))
    if unique:
      groups.append({
          "params": unique,
          "lr": lr,
          "weight_decay": 0.0 if is_normalized else weight_decay,
          "rho": 0.05
      })

  return groups


class setupForData(neuralnetwork.Module):
  def __init__(self, numclasses, rank_classes_dict, freeze_backbones=False):
    super().__init__()
    print("waking up BioCLIP 2 (imageomics/bioclip-2) macro model")
    bioclip_model, _, _ = open_clip.create_model_and_transforms("hf-hub:imageomics/bioclip-2")
    self.path1ViT = bioclip_model.visual
    self.projection1 = neuralnetwork.Sequential(
        neuralnetwork.Linear(768, 256),
        neuralnetwork.LayerNorm(256),
        neuralnetwork.GELU()
    )

    print("waking up textures model")
    ConvMicro = TVmodels.convnext_tiny(weights=TVmodels.ConvNeXt_Tiny_Weights.DEFAULT)
    self.path2CVN = neuralnetwork.Sequential(
        *list(ConvMicro.children())[:-1],
        neuralnetwork.AdaptiveAvgPool2d(1),
        neuralnetwork.Flatten()
    )
    self.projection2 = neuralnetwork.Sequential(
        neuralnetwork.Linear(768, 256),
        neuralnetwork.LayerNorm(256),
        neuralnetwork.GELU()
    )

    print("waking up spatial model")
    ConvSpace = TVmodels.convnext_tiny(weights=TVmodels.ConvNeXt_Tiny_Weights.DEFAULT)
    self.path3CVN = neuralnetwork.Sequential(
        *list(ConvSpace.children())[:-1],
        neuralnetwork.AdaptiveAvgPool2d(1),
        neuralnetwork.Flatten()
    )
    self.projection3 = neuralnetwork.Sequential(
        neuralnetwork.Linear(768, 256),
        neuralnetwork.LayerNorm(256),
        neuralnetwork.GELU()
    )

    if freeze_backbones:
      print("[INFO] Freezing DINOv2 and ConvNeXt backbones.")
      for p in self.path1ViT.parameters():
        p.requires_grad = False
      for p in self.path2CVN.parameters():
        p.requires_grad = False
      for p in self.path3CVN.parameters():
        p.requires_grad = False
    else:
      freeze_bioclip_visual_blocks(
          self.path1ViT,
          freeze_fraction=0.75,
          top_trainable=3
      )

    self.fusion_neck = CrossAttentiveFusion(embed_dim=256, num_heads=4)

    self.species_head = MSDhead(in_features=768, numclasses1=numclasses + 1, numsamples1=5, p=0.2)
    self.family_head = MSDhead(in_features=768, numclasses1=rank_classes_dict["family"], numsamples1=5, p=0.2)
    self.genus_head = MSDhead(in_features=768, numclasses1=rank_classes_dict["genus"], numsamples1=5, p=0.2)
    self.superfamily_head = MSDhead(in_features=768, numclasses1=rank_classes_dict["superfamily"], numsamples1=5, p=0.2)
    self.subfamily_head = MSDhead(in_features=768, numclasses1=rank_classes_dict["subfamily"], numsamples1=5, p=0.2)
    self.infraorder_head = MSDhead(in_features=768, numclasses1=rank_classes_dict["infraorder"], numsamples1=5, p=0.2)

  def forward(self, macroIN, microIN, spatialIN):
    x1 = self.path1ViT(macroIN)
    z1 = self.projection1(x1)

    x2 = self.path2CVN(microIN)
    z2 = self.projection2(x2)

    x3 = self.path3CVN(spatialIN)
    z3 = self.projection3(x3)

    # Zero-guard to prevent blank spatial inputs from polluting attention
    if spatialIN.abs().sum() == 0:
      z3 = torch.zeros_like(z2)

    fused = self.fusion_neck(z1, z2, z3)

    def safe_logits(logits):
      logits = torch.nan_to_num(
          logits,
          nan=0.0,
          posinf=30.0,
          neginf=-30.0
      )
      return logits.clamp(min=-30.0, max=30.0)

    outputs = {
        "species": safe_logits(self.species_head(fused)),
        "family": safe_logits(self.family_head(fused)),
        "genus": safe_logits(self.genus_head(fused)),
        "superfamily": safe_logits(self.superfamily_head(fused)),
        "subfamily": safe_logits(self.subfamily_head(fused)),
        "infraorder": safe_logits(self.infraorder_head(fused)),
        "z1": z1,
        "z2": z2,
        "z3": z3
    }
    return outputs


# Initialize Model & Move to CUDA
CUDA = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = setupForData(numclasses=speciesamt, rank_classes_dict=rank_classes).to(CUDA)

# loss function and optimization
critce = neuralnetwork.CrossEntropyLoss(label_smoothing=0.1)
critce_rank = neuralnetwork.CrossEntropyLoss(label_smoothing=0.1, ignore_index=-1)
ortholossfn = OrthogonalStreamContrastiveLoss(alpha=0.1)

optimizer_groups = build_optimizer_param_groups(
    model,
    model.path1ViT,
    base_lr=1e-5,
    decay=0.75,
    weight_decay=0.01
)
optimizer = SAM(
    optimizer_groups,
    torch.optim.AdamW,
    rho=0.05,
    lr=1e-5,
    weight_decay=0.01
)
scaler = (
    torch.amp.GradScaler("cuda")
    if CUDA.type == "cuda"
    else torch.amp.GradScaler("cpu")
)

# training loop
epochsnum = 25
bestVALacc = 0.0

print(f"Training on {CUDA}")

for epoch in range(epochsnum):
  model.train()
  runningloss = 0.0
  correctpredictions = 0
  totalpredictions = 0

  for batch in loader:
    macroIN2 = batch["macro"].to(CUDA)
    microIN2 = batch["micro"].to(CUDA)
    spatialIN2 = batch["spatial"].to(CUDA)
    labels = batch["label"].to(CUDA)

    optimizer.zero_grad(set_to_none=True)

    with torch.amp.autocast("cuda" if CUDA.type == "cuda" else "cpu"):
      outputs = model(macroIN2, microIN2, spatialIN2)
      lossspecies = critce(outputs["species"], labels)

      losshierarchy = 0.0
      for rank in targetranks:
        ranktarg = [HIERARCHY.get(l, {}).get(rank, -1) for l in labels.tolist()]
        ranktensor = torch.tensor(ranktarg, dtype=torch.long, device=CUDA)
        weight = 0.5 if rank == "genus" else (0.25 if rank == "family" else 0.15)
        losshierarchy += weight * critce_rank(outputs[rank], ranktensor)

      lossortho = ortholossfn(outputs["z1"], outputs["z2"], outputs["z3"])
      loss = lossspecies + losshierarchy + lossortho

    if not torch.isfinite(loss).all():
      print("[WARNING] Non-finite loss detected in initial pass! Skipping batch.")
      optimizer.zero_grad(set_to_none=True)
      continue

    scaler.scale(loss).backward()
    optimizer.first_step(zero_grad=True)

    with torch.amp.autocast("cuda" if CUDA.type == "cuda" else "cpu"):
      outputs_perturbed = model(macroIN2, microIN2, spatialIN2)
      lossspecies_p = critce(outputs_perturbed["species"], labels)

      losshierarchy_p = 0.0
      for rank in targetranks:
        ranktarg = [HIERARCHY.get(l, {}).get(rank, -1) for l in labels.tolist()]
        ranktensor = torch.tensor(ranktarg, dtype=torch.long, device=CUDA)
        weight = 0.5 if rank == "genus" else (0.25 if rank == "family" else 0.15)
        losshierarchy_p += weight * critce_rank(outputs_perturbed[rank], ranktensor)

      lossortho_p = ortholossfn(
          outputs_perturbed["z1"],
          outputs_perturbed["z2"],
          outputs_perturbed["z3"]
      )
      loss_perturbed = lossspecies_p + losshierarchy_p + lossortho_p

    if not torch.isfinite(loss_perturbed).all():
      print("[WARNING] Non-finite loss detected in perturbed pass! Restoring weights.")
      for group in optimizer.param_groups:
        for p in group["params"]:
          old_p = optimizer.state[p].pop("old_p", None)
          if old_p is not None:
            p.copy_(old_p)
      optimizer.zero_grad(set_to_none=True)
      continue

    scaler.scale(loss_perturbed).backward()
    scaler.unscale_(optimizer.base_optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.second_step(zero_grad=True)
    scaler.update()

    runningloss += loss.item() * labels.size(0)
    _, predicts = torch.max(outputs["species"], dim=1)
    correctpredictions += torch.sum(predicts == labels).item()
    totalpredictions += labels.size(0)

  epochloss = runningloss / max(1, totalpredictions)
  epochaccuracy = (correctpredictions / max(1, totalpredictions)) * 100

  # VALIDATION TESTING
  model.eval()
  VALrunningloss = 0.0
  VALcorrect = 0
  VALtotal = 0

  with torch.no_grad():
    for valbatch in validationL:
      valMAC = valbatch["macro"].to(CUDA)
      valMIC = valbatch["micro"].to(CUDA)
      valSPA = valbatch["spatial"].to(CUDA)
      vallabels = valbatch["label"].to(CUDA)

      with torch.amp.autocast("cuda" if CUDA.type == "cuda" else "cpu"):
        VALoutputs = model(valMAC, valMIC, valSPA)
        VALloss = critce(VALoutputs["species"], vallabels)

      if not torch.isfinite(VALloss).all():
        continue

      VALrunningloss += VALloss.item() * vallabels.size(0)
      _, VALpredictions = torch.max(VALoutputs["species"], dim=1)
      VALcorrect += torch.sum(VALpredictions == vallabels).item()
      VALtotal += vallabels.size(0)

  VALloss2 = VALrunningloss / max(1, VALtotal)
  VALaccuracy = (VALcorrect / max(1, VALtotal)) * 100

  if VALaccuracy > bestVALacc:
    bestVALacc = VALaccuracy
    save_file(model.state_dict(), "convnextDINOSAUR_best.safetensors")
    print(f"New best model saved with Val Accuracy: {VALaccuracy:.2f}%")

  print(f"Epoch [{epoch+1}/{epochsnum}] - Training Loss: {epochloss:.4f} | "
        f"Train Acc: {epochaccuracy:.2f}% | Validation Loss: {VALloss2:.4f} | "
        f"Validation Accuracy: {VALaccuracy:.2f}%")


# POST-HOC TEMPERATURE SCALING
class TemperatureScaler(neuralnetwork.Module):
  def __init__(self):
    super().__init__()
    self.log_temperature = neuralnetwork.Parameter(torch.zeros(()))

  @property
  def temperature(self):
    return self.log_temperature.exp().clamp(0.05, 100.0)

  def forward(self, logits):
    return logits / self.temperature

def inverse_frequency_weights(labels, num_classes, valid_mask=None):
  labels = labels.clone().long()
  if valid_mask is not None:
    labels = labels[valid_mask]
  if len(labels) == 0:
    return None

  counts = torch.bincount(labels, minlength=num_classes).float()
  counts = torch.clamp(counts, min=1.0)
  weights = counts.sum() / (num_classes * counts)
  weights = weights / weights.mean()
  return weights

def collect_calibration_logits(model, dataloader, hierarchy, device):
  model.eval()
  collected = defaultdict(list)

  with torch.no_grad():
    for batch in dataloader:
      valMAC = batch["macro"].to(device)
      valMIC = batch["micro"].to(device)
      valSPA = batch["spatial"].to(device)
      specieslabels = batch["label"].to(device)

      with torch.amp.autocast("cuda" if device.type == "cuda" else "cpu"):
        outputs = model(valMAC, valMIC, valSPA)

      collected["species_logits"].append(outputs["species"].cpu())
      collected["species_labels"].append(specieslabels.cpu())

      species_cpu = specieslabels.cpu().tolist()
      for rank in targetranks:
        ranklabels = torch.tensor(
            [hierarchy.get(l, {}).get(rank, -1) for l in species_cpu],
            dtype=torch.long
        )
        collected[f"{rank}_logits"].append(outputs[rank].cpu())
        collected[f"{rank}_labels"].append(ranklabels)

  return {
      key: torch.cat(values, dim=0)
      for key, values in collected.items()
  }

def fit_temperature(logits, labels, weights=None, valid_mask=None):
  logits = logits.detach().float().cpu()
  labels = labels.detach().long().cpu()

  if valid_mask is None:
    valid_mask = torch.ones(labels.shape[0], dtype=torch.bool)

  logits = logits[valid_mask]
  labels = labels[valid_mask]
  if len(labels) == 0:
    return 1.0

  if weights is None:
    weights = inverse_frequency_weights(
        labels,
        logits.shape[1],
        valid_mask=torch.ones_like(valid_mask)
    )
  if weights is None:
    weights = torch.ones(logits.shape[1], dtype=torch.float32)

  weights = weights.to(logits.device)
  sample_weights = weights[labels]
  sample_weights = sample_weights / sample_weights.mean()

  scaler = TemperatureScaler()
  lbfgs = torch.optim.LBFGS(
      scaler.parameters(),
      lr=0.5,
      max_iter=100,
      tolerance_grad=1e-7,
      tolerance_change=1e-9,
      line_search_fn="strong_wolfe"
  )
  criterion = neuralnetwork.CrossEntropyLoss(reduction="none")

  def closure():
    lbfgs.zero_grad()
    losses = criterion(scaler(logits), labels)
    loss = torch.sum(losses * sample_weights) / torch.sum(sample_weights)
    loss.backward()
    return loss

  lbfgs.step(closure)
  temperature = float(scaler.temperature.detach().item())
  if not nump.isfinite(temperature):
    return 1.0
  return temperature

calibration_data = collect_calibration_logits(
    model,
    calibrationL,
    HIERARCHY,
    CUDA
)

temperatures = {}
head_names = ["species"] + targetranks
for head_name in head_names:
  logits = calibration_data[f"{head_name}_logits"]
  labels = calibration_data[f"{head_name}_labels"]
  valid_mask = labels >= 0

  weights = inverse_frequency_weights(
      labels,
      logits.shape[1],
      valid_mask=valid_mask
  )
  temperatures[head_name] = fit_temperature(
      logits,
      labels,
      weights=weights,
      valid_mask=valid_mask
  )

print("Post-hoc temperature scaling:", temperatures)

def calibrate_outputs(raw_outputs, temperatures):
  calibrated = {}
  for head_name in ["species"] + targetranks:
    temperature = float(temperatures[head_name])
    calibrated[head_name] = torch.softmax(
        raw_outputs[head_name] / temperature,
        dim=-1
    )
  return calibrated

def calibrated_species_guess(raw_outputs, temperatures, txnID):
  calibrated = calibrate_outputs(raw_outputs, temperatures)
  confidence, species_index = calibrated["species"].max(dim=-1)
  species_name = (
      fulltrainset.classes[species_index.item()]
      if species_index.item() < len(fulltrainset.classes)
      else None
  )
  return guessing(species_name, confidence.item(), txnID), calibrated


# SAVE FINAL WEIGHTS AFTER ALL EPOCHS FINISH
save_file(model.state_dict(), "convnextDINOSAUR.safetensors")

metadata = {
    "temperatures": temperatures,
    "class_names": fulltrainset.classes,
    "rank_vocabularies": {
        rank: {
            name: index
            for name, index in rankvocabularies[rank].items()
        }
        for rank in targetranks
    },
    "confidence_level": list(confidencelevel)
}
with open("convnextDINOSAUR_metadata.json", "w") as metadata_file:
  json.dump(metadata, metadata_file)

cowsay.trex("saved to convnextDINOSAUR.safetensors")

In [ ]:
# Inference script

import torch
from torchvision import transforms
from PIL import Image
import json
import os
from safetensors.torch import load_file
from google.colab import files

# Load class names
with open("class_names.json", "r") as file:
    class_names = json.load(file)

speciesamt = len(class_names)

# Initialize model architecture structure
CUDA = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inferencemodel = setupForData(numclasses=speciesamt, rank_classes_dict=rank_classes).to(CUDA)

weights_path = "convnextDINOSAUR_best.safetensors"
if os.path.exists(weights_path):
    state_dict = load_file(weights_path)
    inferencemodel.load_state_dict(state_dict)
    print(f"Successfully loaded checkpoint: {weights_path}")
else:
    print(f"{weights_path} not found.")

inferencemodel.eval()

# Define Pristine & Constrained TTA Transforms for Inference
inferencemacro = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

inferencemicro = transforms.Compose([
    transforms.Resize((256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

inferencespatial = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Production Inference & PBDB Fallback Execution Function
def predictSPEC(image_path):
    if not os.path.exists(image_path):
        print(f"Image path {image_path} does not exist.")
        return None

    raw = Image.open(image_path).convert("RGB")

    macro = inferencemacro(raw).unsqueeze(0).to(CUDA)
    micro = inferencemicro(raw).unsqueeze(0).to(CUDA)
    spatial = inferencespatial(raw).unsqueeze(0).to(CUDA)

    with torch.no_grad():
        outputs = inferencemodel(macro, micro, spatial)

        # Species probabilities via Softmax over species head (including +1 Unknown Clade logit)
        logits = outputs["species"]
        probs = torch.softmax(logits, dim=1)
        confscore, predictionindex = torch.max(probs, dim=1)

        confscore = confscore.item()
        predictionindex = predictionindex.item()

        # Check for Open-Set Recognition "Unknown Clade" trigger
        if predictionindex >= len(class_names):
            print("\n[OOD ALERT] Specimen prediction landed on Unknown Clade logit buffer.")
            print("Flagged as an uncataloged or undescribed fossil discovery!")
            return {"status": "NEW_DISCOVERY", "confidence": confscore}

        predictedspeciesraw = class_names[predictionindex]
        formattedspeciesname = predictedspeciesraw.replace("_", " ")
        matchedID = None
        for tid, rec in dinotax2.items():
            if rec["name"] and rec["name"].strip().lower() == formattedspeciesname.lower():
                matchedID = tid
                break

        print("\n--- FIV1 FOSSIL IDENTIFICATION REPORT ---")
        print(f"Raw Model Confidence: {confscore * 100:.2f}%")
        print(f"Top Species Prediction: {formattedspeciesname}")

        finalname, finalscore, taxonomicrank = guessing(formattedspeciesname, confscore, matchedID)

        return {
            "identified_name": finalname,
            "confidence": finalscore,
            "rank": taxonomicrank,
            "raw_logits": outputs
        }


print("Please upload a fossil specimen image for identification:")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\nProcessing uploaded image: {filename}")
    result = predictSPEC(filename)

In [ ]:
# The following cells are for pruning bad folders (aka genera) from the dataset, and restoring them if one is deleted by mistake.

In [ ]:
# for deleting

import os
import shutil
import kagglehub
from google.colab import userdata

remove_list = [                   # enter the names of the genuses you want to remove
    "",

]

for dino in remove_list:
    folder_name = dino.replace(" ", "_")
    folder_path = os.path.join("./fossil-data", folder_name)
    if os.path.exists(folder_path):
        shutil.rmtree(folder_path)
        print(f"Removed local folder: {folder_name}")

kUser = userdata.get('KAGGLE_USERNAME')
kSlug = userdata.get('KAGGLE_SLUG')
dataDir = "./fossil-data"

handle = f"{kUser}/{kSlug}"
print(f"Pushing updated dataset to: {handle}")
kagglehub.dataset_upload(
    handle=handle,
    local_dataset_dir=dataDir,
    version_notes="Pruned bad species"
)
print("Upload complete!")

In [ ]:
# for restoring

import os
import shutil
import kagglehub
from google.colab import userdata

# list all the genera you want to reset/rescrape here:
target_species = [""]

for dino in target_species:
    print(f"Processing: {dino}")
    bad_folder = os.path.join(dataDir, dino)

    # 1. remove local folder if it exists (just in case measure)
    if os.path.exists(bad_folder):
        shutil.rmtree(bad_folder)
        print(f"-> Removed {dino} from local data.")

    # 2. rescrape and filter
    print(f"-> Rescraping {dino}...")
    scrapeDino(dino, mIMperSP)
    filterIM(dino)

# 3.push the fully updated dataset to Kaggle once all are done
kUser = userdata.get('KAGGLE_USERNAME')
kSlug = userdata.get('KAGGLE_SLUG')
handle = f"{kUser}/{kSlug}"

print(f"Pushing corrected dataset to: {handle}")
kagglehub.dataset_upload(
    handle=handle,
    local_dataset_dir=dataDir,
    version_notes="new"
)
print("Upload complete")

In [ ]:
# The following cells are for debugging cell 4; use if you recieve a 401 Unauthorized error from Google Kaggle. Otherwise, use Colab secrets.

In [ ]:
import os
import shutil

os.makedirs('/root/.kaggle', exist_ok=True)
if os.path.exists('kaggle.json'):
    shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 600)
    print("New kaggle.json successfully installed!")
else:
    print("Please upload your kaggle.json file using the widget above.")

In [ ]:
from google.colab import files
files.upload()

In [ ]:
# Confusion Matrix; this will show you which species of dinosaur the model is most confused with.
# This cell outputs a graph along with the 10 most confused species pairs. Run this at the end of training.

import seaborn as sb
import matplotlib.pyplot as mpl
from sklearn.metrics import confusion_matrix
import numpy as np
import torch
from safetensors.torch import load_file
import os
import json

# Ensure CUDA and model instances are active
CUDA = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Quick check if model is defined; if not, re-initialize it
if 'model' not in globals():
    print("[INFO] Initializing model instance for evaluation...")
    with open("class_names.json", "r") as f:
        class_names_loaded = json.load(f)
    speciesamt = len(class_names_loaded)
    model = setupForData(numclasses=speciesamt, rank_classes_dict=rank_classes).to(CUDA)

# Load the best saved weights (using the correct filename)
weights_path = "convnextDINOSAUR_best.safetensors"
if os.path.exists(weights_path):
    model.load_state_dict(load_file(weights_path))
    print(f"Successfully loaded weights from {weights_path}")
else:
    print(f"Warning: {weights_path} not found. Ensure training has completed and saved weights.")

model.eval()

predictions = []
true_labels = []

print("Running validation inference for confusion matrix...")
with torch.no_grad():
    for batch in validationL:
        macroIN = batch["macro"].to(CUDA)
        microIN = batch["micro"].to(CUDA)
        spatialIN = batch["spatial"].to(CUDA)
        batchlabels = batch["label"].to(CUDA)

        # Forward pass through triple-stream architecture
        outputs = model(macroIN, microIN, spatialIN)
        species_logits = outputs["species"]

        _, preds = torch.max(species_logits, dim=1)

        # Clamp predictions to valid species range to prevent index out-of-bounds errors
        preds = torch.clamp(preds, max=len(fulltrainset.classes) - 1)

        predictions.extend(preds.cpu().numpy())
        true_labels.extend(batchlabels.cpu().numpy())

# Generate confusion matrix across valid species classes
all_class_indices = list(range(len(fulltrainset.classes)))
matrix = confusion_matrix(true_labels, predictions, labels=all_class_indices)

# Plot Heatmap
mpl.figure(figsize=(14, 12))
sb.heatmap(
    matrix,
    annot=False,
    cmap='Blues',
    fmt='g',
    xticklabels=fulltrainset.classes,
    yticklabels=fulltrainset.classes
)
mpl.title('Species Confusion Matrix (Best Model)')
mpl.ylabel('Actual Species')
mpl.xlabel('Predicted Species')
mpl.xticks(rotation=90, fontsize=8)
mpl.yticks(fontsize=8)
mpl.tight_layout()
mpl.show()

# Extract and print top confused species pairs
class_names = fulltrainset.classes
matrix_temp = matrix.copy()
np.fill_diagonal(matrix_temp, 0)
confusedindices = np.unravel_index(np.argsort(matrix_temp.ravel())[::-1], matrix_temp.shape)

print("\nTOP 10 MOST CONFUSED SPECIES PAIRS")
printed_count = 0
for i in range(len(confusedindices[0])):
    if printed_count >= 10:
        break
    actualidx = confusedindices[0][i]
    predidx = confusedindices[1][i]
    count = matrix_temp[actualidx, predidx]
    if count > 0:
        printed_count += 1
        print(f"{printed_count}. Actual: '{class_names[actualidx]}' → Mistaken for: '{class_names[predidx]}' ({count} times)")